In [0]:
#df_traffic_clean = (spark.readStream.table('traffic_data.bronze.raw_traffic'))

### read stream

In [0]:
def reading_raw_traffic():
    return spark.readStream.table('traffic_data.bronze.raw_traffic')

###Remove Duplicates

In [0]:
def remove_dup(df):
    df_clean = df.dropDuplicates()
    print('Duplicates Removed')
    return df

### Replace Null

In [0]:
def replace_null(df):
    columns = df.schema.names
    print('Remove Null with Unknown for string')
    df1 = df.fillna('Unkown', columns )
    print('Remove Null with 0 for interger')
    df_clean = df1.fillna(0,columns)
    return df_clean
    

### Electric Vehicle Count

In [0]:
def count_EV(df):
    from pyspark.sql.functions import col
    df_clean = df.withColumn('Electric_vehicle_count', col('EV_Car')+col('EV_Bike'))
    print('column Electric_vehicle_count added')
    return df_clean

###Motor vehicle count

In [0]:
def count_motor_vehicle(df):
    from pyspark.sql.functions import col
    df_clean = df.withColumn('Motor_vehicle_count', 
                             col('EV_Car')+col('EV_Bike')
                             + col('Two_wheeled_motor_vehicles')
                             +col('Cars_and_taxis')
                             +col('Buses_and_coaches')
                             +col('LGV_Type')+col('HGV_Type')
                             )
    print('column Motor vehicle count added')
    return df_clean

### Add transformed Time Column


In [0]:
def add_transformed_time(df):
    from pyspark.sql.functions import current_timestamp
    print('Adding Silver_Transformed_Time column')
    return df.withColumn('Silver_Transformed_Time',current_timestamp())

### Write Stream

In [0]:
def write_stream(df):
    source = spark.sql('DESCRIBE EXTERNAL LOCATION trafficprojectexternallocation').select('url').collect()[0][0]
    ws = (
        df.writeStream
          .format('delta')
          .option('CheckpointLocation',f'{source}/Checkpoint/Silver/TrafficLoad')
          .outputMode('append')
          .queryName('silverTrafficLoad')
          .trigger(availableNow = True)
          .toTable('traffic_data.silver.trafficLoad')
    )

    ws.awaitTermination()
    print('Write Completed')

### Function Call

In [0]:
df1 = reading_raw_traffic()
df2 = remove_dup(df1)
df3 = replace_null(df2)
df4 = count_EV(df3)
df5 = count_motor_vehicle(df4)
df6 = add_transformed_time(df5)
#display(df6)
write_stream(df6)

In [0]:
%sql
select * from traffic_data.silver.trafficLoad;

In [0]:
%sql
select * from traffic_data.silver.trafficLoad where Record_ID between 18540 and 18560 order by Record_ID;